# Test evaluation

## 1. Final scores on the test set
All choices were made on validation. I evaluate every model once on the test set (January 2024 to August 2026) and change nothing afterwards.
The Elo logistic and boosting models are refitted on train + validation with the settings chosen on validation. Poisson and Dixon-Coles are refitted every month on all earlier matches, as before.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd

from src.database import ROOT, query
from src.evaluation import OUTCOMES, compare, paired_difference
from src.models.boosting import BoostingModel
from src.models.dixon_coles import DixonColesModel
from src.models.elo_logistic import EloLogistic
from src.models.poisson import PoissonModel, rolling_predict
from src.splits import load_matches

expected_goals = pd.read_csv(ROOT / "data" / "processed" / "poisson_expected_goals.csv")
matches = load_matches().merge(expected_goals, on="match_id")
fit_set = matches[matches["split"] != "test"]
test = matches[matches["split"] == "test"]
history = query("""
    SELECT date, home_team, away_team, home_score, away_score, neutral
    FROM clean_results
    ORDER BY date, match_id
""")
len(fit_set), len(test)

(21377, 2583)

In [2]:
frequencies = fit_set.groupby("neutral")["result"].value_counts(normalize=True).unstack()[OUTCOMES]
stacking_features = ["elo_diff", "home_venue", "poisson_home_goals", "poisson_away_goals"]

test_probs = {
    "uniform": np.full((len(test), 3), 1 / 3),
    "frequency_by_venue": frequencies.reindex(test["neutral"]).to_numpy(),
    "elo_logistic": EloLogistic().fit(fit_set).predict_proba(test),
    "boosting": BoostingModel(max_iter=50, max_depth=4).fit(fit_set).predict_proba(test),
    "poisson": rolling_predict(lambda: PoissonModel(), history, test),
    "dixon_coles": rolling_predict(lambda: DixonColesModel(), history, test),
    "stacking": BoostingModel(max_iter=50, max_depth=4, features=stacking_features).fit(fit_set).predict_proba(test),
}
compare(test_probs, test["result"])

,log_loss,brier,rps
uniform,1.0986,0.6667,0.2372
frequency_by_venue,1.0538,0.6352,0.2256
elo_logistic,0.8649,0.5098,0.1658
boosting,0.8652,0.5091,0.1657
poisson,0.8448,0.4963,0.1602
dixon_coles,0.8432,0.4957,0.1601
stacking,0.8478,0.4983,0.1610


In [3]:
pairs = [("poisson", "elo_logistic"), ("stacking", "poisson"), ("boosting", "elo_logistic")]
pd.DataFrame(
    [{"comparison": f"{a} - {b}", **paired_difference(test_probs[a], test_probs[b], test["result"])} for a, b in pairs]
).round(4)

,comparison,mean_diff,std_error
0,poisson - elo_logistic,-0.0202,0.0048
1,stacking - poisson,0.0030,0.0027
2,boosting - elo_logistic,0.0003,0.0027


The test confirms the validation ranking. Poisson beats the Elo baseline by 0.0202 in log loss (4.2 standard errors), a clear gap that was only borderline on validation. Boosting alone adds nothing to the Elo baseline. Stacking and Dixon-Coles are within noise of Poisson, so the simplest model stays the right choice.
All scores are better than on validation, for every model including the baselines: 2024-2026 matches were slightly easier to predict. The ranking of models is what matters.

## 2. Error analysis of the final model
I look at the per-match log loss of Poisson by competition type and by Elo gap, then at its biggest surprises.

In [4]:
from src.evaluation import log_loss_per_match, one_hot

poisson_probs = test_probs["poisson"]
errors = test.assign(
    log_loss_poisson=log_loss_per_match(poisson_probs, test["result"]),
    log_loss_elo=log_loss_per_match(test_probs["elo_logistic"], test["result"]),
    p_actual=np.sum(poisson_probs * one_hot(test["result"]), axis=1),
    p_home=poisson_probs[:, 0], p_draw=poisson_probs[:, 1], p_away=poisson_probs[:, 2],
    is_draw=test["result"] == "D",
)
major = ["FIFA World Cup", "UEFA Euro", "Copa América", "African Cup of Nations", "AFC Asian Cup", "Gold Cup"]
errors["competition"] = np.select(
    [errors["tournament"] == "Friendly", errors["tournament"].isin(major)],
    ["friendly", "major_tournament"],
    default="qualifier_or_league",
)
errors.groupby("competition").agg(
    n_matches=("result", "size"),
    log_loss_poisson=("log_loss_poisson", "mean"),
    log_loss_elo=("log_loss_elo", "mean"),
    draw_rate=("is_draw", "mean"),
).round(4)

,n_matches,log_loss_poisson,log_loss_elo,draw_rate
competition,,,,
friendly,662,0.8791,0.8873,0.2508
major_tournament,370,0.9106,0.9292,0.3189
qualifier_or_league,1551,0.8144,0.8401,0.2224


In [5]:
errors["elo_gap"] = pd.cut(errors["elo_diff"].abs(), [0, 50, 100, 200, 400, 2000], right=False)
errors.groupby("elo_gap", observed=True).agg(
    n_matches=("result", "size"),
    predicted_draw=("p_draw", "mean"),
    observed_draw=("is_draw", "mean"),
).round(4)

,n_matches,predicted_draw,observed_draw
elo_gap,,,
"[0, 50)",371,0.2848,0.3261
"[50, 100)",367,0.2776,0.2888
"[100, 200)",674,0.2550,0.2908
"[200, 400)",812,0.1972,0.2180
"[400, 2000)",359,0.1002,0.0808


In [6]:
errors.nsmallest(10, "p_actual")[
    ["date", "home_team", "away_team", "home_score", "away_score", "tournament", "p_home", "p_draw", "p_away"]
].round({"p_home": 3, "p_draw": 3, "p_away": 3})

,date,home_team,away_team,home_score,away_score,tournament,p_home,p_draw,p_away
22444,2024-11-17,Thailand,Laos,1,1,Friendly,0.990,0.008,0.002
23770,2026-06-06,Bermuda,Cape Verde,3,0,Friendly,0.013,0.038,0.948
21766,2024-06-07,England,Iceland,0,1,Friendly,0.899,0.074,0.027
22530,2024-12-12,Indonesia,Laos,3,3,ASEAN Championship,0.964,0.028,0.008
22111,2024-09-09,Samoa,Tonga,1,1,FIFA World Cup qualification,0.954,0.031,0.015
21776,2024-06-07,Romania,Liechtenstein,0,0,Friendly,0.963,0.033,0.004
23384,2025-11-17,Bahrain,Somalia,1,2,Friendly,0.795,0.160,0.045
22770,2025-05-30,Saint Vincent and the Grenadines,British Virgin Islands,1,1,Friendly,0.937,0.049,0.014
22000,2024-09-05,Australia,Bahrain,0,1,FIFA World Cup qualification,0.809,0.137,0.054
21725,2024-06-06,Afghanistan,Qatar,0,0,FIFA World Cup qualification,0.018,0.056,0.926


In [7]:
favourite = np.maximum(errors["p_home"], errors["p_away"])
favourite_won = np.where(errors["p_home"] >= errors["p_away"], errors["result"] == "H", errors["result"] == "A")
bins = pd.cut(favourite, [0.8, 0.9, 0.95, 1.0], right=False)
pd.DataFrame({"announced": favourite, "won": favourite_won}).groupby(bins, observed=True).agg(
    n_matches=("won", "size"), announced=("announced", "mean"), won=("won", "mean"),
).round(3)

,n_matches,announced,won
"[0.8, 0.9)",259,0.846,0.876
"[0.9, 0.95)",103,0.924,0.932
"[0.95, 1.0)",89,0.973,0.955


- Major tournaments are the hardest to forecast (log loss 0.911 vs 0.814 for qualifiers): teams are closer and a third of games end in a draw.
- Poisson beats the Elo baseline in every competition type.
- Main weakness: draws in balanced games. Between close teams, Poisson predicts 28.5% draws for 32.6% observed (22.3% vs 24.4% overall).
- Next lead: a bivariate Poisson with an inflated diagonal.
- The biggest surprises are upsets of small teams, often in friendlies. They are expected chance events: when the favourite gets 97% on average, it wins 95.5% of the time.